# Trino’s iceberg connector selects the HMS catalog in iceberg.properties and delta.properties

In [1]:
%cd ~/work/on-premises/docker

/home/jovyan/work/on-premises/docker


In [2]:
!echo "=== Delta ==="
!cat trino/catalog/delta.properties
!echo === Iceberg ===
!cat trino/catalog/iceberg.properties
!echo === Init catalog ==
!cat hive-metastore/init-catalogs.sh

=== Delta ===
connector.name=delta_lake
hive.metastore.uri=thrift://hive-metastore:9083
hive.metastore.thrift.catalog-name=delta
fs.s3.enabled=true
s3.endpoint=http://seaweedfs:8334
s3.region=us-east-1
s3.path-style-access=true
s3.aws-access-key=${ENV:SEAWEEDFS_ACCESS_KEY_ID}
s3.aws-secret-key=${ENV:SEAWEEDFS_SECRET_ACCESS_KEY}
=== Iceberg ===
connector.name=iceberg
iceberg.catalog.type=hive_metastore
hive.metastore.uri=thrift://hive-metastore:9083
hive.metastore.thrift.catalog-name=iceberg
fs.s3.enabled=true
s3.endpoint=http://seaweedfs:8334
s3.region=us-east-1
s3.path-style-access=true
s3.aws-access-key=${ENV:SEAWEEDFS_ACCESS_KEY_ID}
s3.aws-secret-key=${ENV:SEAWEEDFS_SECRET_ACCESS_KEY}
=== Init catalog ==
#!/usr/bin/env bash
set -o errexit -o nounset -o pipefail

hive_home=${HIVE_HOME:-/opt/hive}
hive_conf_dir=${hive_home}/conf
custom_conf_dir=${HIVE_CUSTOM_CONF_DIR:-/etc/hive/postgres/conf}
db_driver=${DB_DRIVER:-postgres}

find "${custom_conf_dir}" -type f -exec ln -sfn {} "${hive_

In [3]:
%%tsql
-- Trino’s iceberg connector selects the HMS catalog in iceberg.properties and delta.properties
SHOW CATALOGS;

SELECT catalog_name, connector_name
FROM system.metadata.catalogs
ORDER BY catalog_name;


SELECT table_catalog, table_schema, table_name, table_type
FROM iceberg.information_schema.tables
ORDER BY table_schema, table_name;

SELECT table_schema, table_name
FROM iceberg.system.iceberg_tables
ORDER BY table_schema, table_name;

,Catalog
0,delta
1,iceberg
2,system


,catalog_name,connector_name
0,delta,delta_lake
1,iceberg,iceberg
2,system,system


,table_catalog,table_schema,table_name,table_type
0,iceberg,information_schema,applicable_roles,BASE TABLE
1,iceberg,information_schema,columns,BASE TABLE
2,iceberg,information_schema,enabled_roles,BASE TABLE
3,iceberg,information_schema,roles,BASE TABLE
4,iceberg,information_schema,schemata,BASE TABLE
5,iceberg,information_schema,table_privileges,BASE TABLE
6,iceberg,information_schema,tables,BASE TABLE
7,iceberg,information_schema,views,BASE TABLE
8,iceberg,nyc_gov,country_codes,BASE TABLE
9,iceberg,nyc_gov,given_name_frequency,BASE TABLE


TrinoExternalError: TrinoExternalError(type=EXTERNAL, name=HIVE_METASTORE_ERROR, message="Could not find database hive.nyc_gov", query_id=20260916_203926_00003_abst9)

# Hive metastore - Postgres

In [4]:
%%bash
docker compose exec -T postgres \
  psql -U postgres -d hive_metastore -v ON_ERROR_STOP=1 <<'SQL'
SELECT "NAME", "OWNER_NAME",  "OWNER_TYPE", "DB_LOCATION_URI", "TYPE", "CTLG_NAME", "DESC"
FROM "DBS"
SQL


docker compose exec -T postgres \
  psql -U postgres -d hive_metastore -v ON_ERROR_STOP=1 <<'SQL'
SELECT "TBL_NAME", "TBL_TYPE", "OWNER", "OWNER_TYPE"
FROM "TBLS"
SQL

  NAME   | OWNER_NAME | OWNER_TYPE |    DB_LOCATION_URI    |  TYPE  | CTLG_NAME |         DESC          
---------+------------+------------+-----------------------+--------+-----------+-----------------------
 default | public     | ROLE       | s3a://trino-lakehouse | NATIVE | hive      | Default Hive database
(1 row)

 TBL_NAME | TBL_TYPE | OWNER | OWNER_TYPE 
----------+----------+-------+------------
(0 rows)



# Hive metastore beeline - list tables

In [5]:
%%bash
docker compose exec -T hive-metastore \
  script -qec "beeline --silent=true --showHeader=true --outputformat=table -u 'jdbc:hive2://hive-server2:10000/default' -n hive -e '!tables'" \
  /dev/null

+------------+--------------+-------------+-------------+----------+-----------+-------------+------------+----------------------------+-----------------+
| TABLE_CAT  | TABLE_SCHEM  | TABLE_NAME  | TABLE_TYPE  | REMARKS  | TYPE_CAT  | TYPE_SCHEM  | TYPE_NAME  | SELF_REFERENCING_COL_NAME  | REF_GENERATION  |
+------------+--------------+-------------+-------------+----------+-----------+-------------+------------+----------------------------+-----------------+
+------------+--------------+-------------+-------------+----------+-----------+-------------+------------+----------------------------+-----------------+


# Hive metastore beeline - drop all tables

In [6]:
%%bash
docker compose exec -T hive-metastore \
  script -qec "beeline \
    --silent=false \
    --showHeader=true \
    --outputformat=table \
    -u 'jdbc:hive2://hive-server2:10000/default' \
    -n hive \
    -e 'DROP TABLE nyc_gov.given_name_frequency;  DROP TABLE nyc_gov.name_equivalence; DROP TABLE nyc_gov.popular_names_by_country_2026'" \
  /dev/null

Connecting to jdbc:hive2://hive-server2:10000/default
Connected to: Apache Hive (version 4.2.1)
Driver: Hive JDBC (version 4.2.1)
Transaction isolation: TRANSACTION_REPEATABLE_READ
INFO  : Compiling command(queryId=hive_20260916184253_f40a1992-a234-4543-bb01-8a551dba6591): DROP TABLE nyc_gov.given_name_frequency
INFO  : Semantic Analysis Completed (retrial = false)
INFO  : Created Hive schema: Schema(fieldSchemas:null, properties:null)
INFO  : Completed compiling command(queryId=hive_20260916184253_f40a1992-a234-4543-bb01-8a551dba6591); Time taken: 0.562 seconds
INFO  : Executing command(queryId=hive_20260916184253_f40a1992-a234-4543-bb01-8a551dba6591): DROP TABLE nyc_gov.given_name_frequency
INFO  : Starting task [Stage-0:DDL] in serial mode
INFO  : Completed executing command(queryId=hive_20260916184253_f40a1992-a234-4543-bb01-8a551dba6591); Time taken: 0.014 seconds
No rows affected (0.654 seconds)
INFO  : Compiling command(queryId=hive_20260916184254_62bdaa19-77af-41f9-8ded-6fafcd3

# View Postgres records

In [2]:
%%bash
docker compose exec -T postgres \
  psql -U postgres -d hive_metastore -v ON_ERROR_STOP=1 <<'SQL'
SELECT "NAME", "OWNER_NAME",  "OWNER_TYPE", "DB_LOCATION_URI", "TYPE", "CTLG_NAME", "DESC"
FROM "DBS"
SQL

  NAME   | OWNER_NAME | OWNER_TYPE |    DB_LOCATION_URI    |  TYPE  | CTLG_NAME |         DESC          
---------+------------+------------+-----------------------+--------+-----------+-----------------------
 default | public     | ROLE       | s3a://trino-lakehouse | NATIVE | hive      | Default Hive database
(1 row)



# Docker compose list containers

In [52]:
!docker compose ps

NAME                  IMAGE                                       COMMAND                  SERVICE             CREATED          STATUS                    PORTS
airflow-2.11          local/airflow:2.11.2                        "/usr/bin/dumb-init …"   airflow-2.11        58 minutes ago   Up 58 minutes (healthy)   8080/tcp
airflow-3.3           local/airflow:3.3.0                         "/usr/bin/dumb-init …"   airflow-3.3         58 minutes ago   Up 58 minutes (healthy)   8080/tcp
debezium              quay.io/debezium/connect:3.3.2.Final        "/docker-entrypoint.…"   debezium            58 minutes ago   Up 57 minutes (healthy)   8778/tcp, 127.0.0.1:8083->8083/tcp, 9092/tcp
deepseek-harness      local/deepseek-harness:0.1.6-alpha.1-r1     "/usr/bin/tini -- /u…"   deepseek-harness    58 minutes ago   Up 58 minutes (healthy)   3080/tcp, 127.0.0.1:6080->6080/tcp
flink-jobmanager      local/flink:2.1.3-delta-4.4.0               "/docker-entrypoint.…"   flink-jobmanager    58 minutes ago 

# Stop Trino

In [58]:
!docker compose stop trino

[+] stop 0/1
 ⠋ Container trino Stopping                                                 0.0s
[+] stop 0/1
 ⠙ Container trino Stopping                                                 0.1s
[+] stop 0/1
 ⠹ Container trino Stopping                                                 0.2s
[+] stop 1/1
 ✔ Container trino Stopped                                                  0.3s


# Drop catalogs Delta and Iceberg
*There are no tables in the Hive Metastore, so it is safe to remove the catalog.*

In [18]:
%%bash
docker compose  exec -T hive-metastore \
  script -qec "beeline --silent=true \
  -u 'jdbc:hive2://hive-server2:10000/default' -n hive \
  -e 'DROP DATABASE IF EXISTS nyc_gov; DROP DATABASE IF EXISTS test;'" \
  /dev/null

In [23]:
%%bash
docker compose  exec -T hive-metastore \
  script -qec "beeline --silent=true \
  -u 'jdbc:hive2://hive-server2:10000/default' -n hive \
  -e 'DROP CATALOG IF EXISTS iceberg; DROP CATALOG IF EXISTS delta;'" \
  /dev/null

In [24]:
%%bash
docker compose  exec -T hive-metastore \
  script -qec "beeline --silent=true \
  -u 'jdbc:hive2://hive-server2:10000/default' -n hive \
  -e 'SHOW CATALOGS; SHOW DATABASES; SHOW SCHEMAS'" \
  /dev/null

+---------------+
| catalog_name  |
+---------------+
| hive          |
+---------------+
+----------------+
| database_name  |
+----------------+
| default        |
+----------------+
+----------------+
| database_name  |
+----------------+
| default        |
+----------------+


In [17]:
!echo "=== Delta ==="
!cat trino/catalog/delta.properties
!echo === Iceberg ===
!cat trino/catalog/iceberg.properties
!echo === Init catalog ==
!cat hive-metastore/init-catalogs.sh

=== Delta ===
connector.name=delta_lake
hive.metastore.uri=thrift://hive-metastore:9083
hive.metastore.thrift.catalog-name=delta
fs.s3.enabled=true
s3.endpoint=http://seaweedfs:8334
s3.region=us-east-1
s3.path-style-access=true
s3.aws-access-key=${ENV:SEAWEEDFS_ACCESS_KEY_ID}
s3.aws-secret-key=${ENV:SEAWEEDFS_SECRET_ACCESS_KEY}
=== Iceberg ===
connector.name=iceberg
iceberg.catalog.type=hive_metastore
hive.metastore.uri=thrift://hive-metastore:9083
hive.metastore.thrift.catalog-name=iceberg
fs.s3.enabled=true
s3.endpoint=http://seaweedfs:8334
s3.region=us-east-1
s3.path-style-access=true
s3.aws-access-key=${ENV:SEAWEEDFS_ACCESS_KEY_ID}
s3.aws-secret-key=${ENV:SEAWEEDFS_SECRET_ACCESS_KEY}
=== Init catalog ==
#!/usr/bin/env bash
set -o errexit -o nounset -o pipefail

hive_home=${HIVE_HOME:-/opt/hive}
hive_conf_dir=${hive_home}/conf
custom_conf_dir=${HIVE_CUSTOM_CONF_DIR:-/etc/hive/postgres/conf}
db_driver=${DB_DRIVER:-postgres}

find "${custom_conf_dir}" -type f -exec ln -sfn {} "${hive_

In [15]:
%%bash
docker compose  exec -T hive-metastore \
  script -qec "beeline --silent=true \
  -u 'jdbc:hive2://hive-server2:10000/default' -n hive \
  -e 'DESCRIBE CATALOG EXTENDED iceberg; DESCRIBE CATALOG EXTENDED delta; DESCRIBE DATABASE EXTENDED default;'" \
  /dev/null

+---------------+--------------------------+-----------+--------------+
|   cat_name    |         comment          | location  | create_time  |
+---------------+--------------------------+-----------+--------------+
| Catalog Name  | iceberg                  | NULL      | NULL         |
| Comment       | Iceberg warehouse        | NULL      | NULL         |
| Location      | s3a://iceberg-lakehouse  | NULL      | NULL         |
+---------------+--------------------------+-----------+--------------+
+---------------+------------------------+-----------+--------------+
|   cat_name    |        comment         | location  | create_time  |
+---------------+------------------------+-----------+--------------+
| Catalog Name  | delta                  | NULL      | NULL         |
| Comment       | Delta Lake warehouse   | NULL      | NULL         |
| Location      | s3a://delta-lakehouse  | NULL      | NULL         |
+---------------+------------------------+-----------+--------------+
+-----

# Start Trino

In [56]:
!docker compose start trino

[+] start 0/1
 ⠋ Container startup Starting                                               0.0s
[+] start 0/1
 ⠙ Container startup Waiting                                                0.1s
[+] start 0/1
 ⠹ Container startup Waiting                                                0.2s
[+] start 0/1
 ⠸ Container startup Waiting                                                0.3s
[+] start 0/1
 ⠼ Container startup Waiting                                                0.4s
[+] start 0/1
 ⠴ Container startup Waiting                                                0.5s
[+] start 1/3
 ✔ Container startup   Exited                                               0.6s
 ⠋ Container seaweedfs Waiting                                              0.1s
 ⠋ Container postgres  Waiting                                              0.1s
[+] start 1/3
 ✔ Container startup   Exited                                               0.6s
 ⠙ Container seaweedfs Waiting                                              0.

# Check

In [14]:
%%tsql
SELECT catalog_name, connector_name
FROM system.metadata.catalogs
ORDER BY catalog_name;


SELECT table_catalog, table_schema, table_name, table_type
FROM iceberg.information_schema.tables
ORDER BY table_schema, table_name

,catalog_name,connector_name
0,delta,delta_lake
1,iceberg,iceberg
2,system,system


,table_catalog,table_schema,table_name,table_type
0,iceberg,information_schema,applicable_roles,BASE TABLE
1,iceberg,information_schema,columns,BASE TABLE
2,iceberg,information_schema,enabled_roles,BASE TABLE
3,iceberg,information_schema,roles,BASE TABLE
4,iceberg,information_schema,schemata,BASE TABLE
5,iceberg,information_schema,table_privileges,BASE TABLE
6,iceberg,information_schema,tables,BASE TABLE
7,iceberg,information_schema,views,BASE TABLE
8,iceberg,system,iceberg_tables,BASE TABLE


# Trino

In [11]:
%tsql  SHOW CATALOGS;

,Catalog
0,delta
1,iceberg
2,system


In [12]:
%tsql SHOW SCHEMAS FROM iceberg;

,Schema
0,information_schema
1,system


In [13]:
%tsql SHOW SCHEMAS FROM delta;

,Schema
0,information_schema


In [14]:
%tsql SHOW SCHEMAS FROM system;

,Schema
0,information_schema
1,jdbc
2,metadata
3,runtime
